In [1]:
# 필요한 라이브러리 불러오기
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score
from sklearn.metrics import accuracy_score

In [2]:
# 데이터 불러오기
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [4]:
# 데이터 전처리
df.isnull().sum()

# categorical
categorical = df.dtypes[df.dtypes == 'object'].index.to_list()

# numerical
numerical = df.dtypes[df.dtypes != 'object'].index.to_list()


df[categorical]=df[categorical].fillna('NA')
df[numerical]=df[numerical].fillna(0)

**Question1**

In [5]:
df['industry'].mode()

0    technology
Name: industry, dtype: object

**Question2**

In [6]:
df[numerical].corr() # interaction X lead_score

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [7]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [8]:
X_train=df_train.drop('converted',axis=1)
y_train=df_train['converted']

X_test=df_test.drop('converted',axis=1)
y_test=df_test['converted']

X_val=df_val.drop('converted',axis=1)
y_val=df_val['converted']

**Question3**

In [9]:
# mutual_information : 한 변수의 값을 알았을 떄 다른 변수의 불확실성이 얼마나 줄어드는지 
print(mutual_info_score(df['converted'],df['industry']))
print(mutual_info_score(df['converted'],df['location']))
print(mutual_info_score(df['converted'],df['lead_source'])) # converted, lead_source 
print(mutual_info_score(df['converted'],df['employment_status']))

0.0022800937730549087
0.0010636898089210062
0.033063899391404215
0.022797267862555814


**Quesition4**

In [10]:
# categorical, numerical에는 입력 feature 열 이름만 넣으세요.
# 정답 열 'converted'는 numerical에 포함하지 않습니다.
numerical = [col for col in numerical if col != "converted"]

ohe = OneHotEncoder(handle_unknown="ignore")

# 범주형 열 인코딩
train_encoded = pd.DataFrame(
    ohe.fit_transform(X_train[categorical]).toarray(),
    columns=ohe.get_feature_names_out(categorical),
    index=X_train.index
)

val_encoded = pd.DataFrame(
    ohe.transform(X_val[categorical]).toarray(),
    columns=ohe.get_feature_names_out(categorical),
    index=X_val.index
)

X_train_final = pd.concat([X_train[numerical], train_encoded], axis=1)
X_val_final = pd.concat([X_val[numerical], val_encoded], axis=1)

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)
model.fit(X_train_final, y_train)

y_pred = model.predict(X_val_final)
accuracy = accuracy_score(y_val, y_pred)

print(f"Validation accuracy: {accuracy:.2f}")

Validation accuracy: 0.65


**Question5**

In [11]:

features = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

results = {}

for feature in features:
    # 제거할 열 선택
    if feature in categorical:
        cols = [col for col in X_train_final.columns
                if col.startswith(f"{feature}_")]
    else:
        cols = [feature]

    # Feature 제거
    X_train_drop = X_train_final.drop(columns=cols)
    X_val_drop = X_val_final.drop(columns=cols)

    # 모델 학습
    model = LogisticRegression( 
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_drop, y_train)
    y_pred = model.predict(X_val_drop)

    # Accuracy 차이 계산
    new_accuracy = accuracy_score(y_val, y_pred)
    results[feature] = accuracy - new_accuracy

    print(f"{feature}: {results[feature]:.6f}")

print("정답:", min(results, key=results.get))


lead_source: 0.003000
number_of_courses_viewed: 0.002000
interaction_count: 0.044000
정답: number_of_courses_viewed


**Question6**

In [12]:
# 규제항 규제
for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(
        solver="liblinear",
        C=c,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train_final, y_train)

    y_pred = model.predict(X_val_final)
    accuracy = accuracy_score(y_val, y_pred)
    
    # C:0.001,accuracy:0.645 >> 0.65
    print(f'C:{c},accuracy:{accuracy}') 

C:1e-06,accuracy:0.598
C:1e-05,accuracy:0.598
C:0.0001,accuracy:0.613
C:0.001,accuracy:0.645
